# 01 Data Ingestion Orchestrator (TPC-H to Iceberg via Trino)

Notebook ini berfungsi sebagai orkestrator untuk memindahkan data mentah TPC-H (CSV) ke dalam arsitektur Data Lakehouse (Apache Iceberg) menggunakan **Trino** sebagai *query engine*.

**Langkah-langkah yang akan dijalankan:**
1. **Prerequisite & Sanity Check:** Memastikan Docker (MinIO, Hive, Trino) berjalan.
2. **Upload CSV:** Memuat data CSV lokal ke bucket MinIO.
3. **Cleanup:** Membersihkan sisa data Iceberg sebelumnya agar tidak terjadi duplikasi.
4. **Execute Trino SQL:** Mengeksekusi file `tpch_iceberg_schema.sql` untuk membuat skema, *external table*, dan tabel Iceberg.
5. **Validation:** Menghitung jumlah baris data yang berhasil masuk ke Iceberg.

In [14]:
import sys
import subprocess
import time
from pathlib import Path

try:
    from minio import Minio
    from trino.dbapi import connect
    print("Library 'minio' dan 'trino' siap digunakan")
except ImportError:
    print("Menginstal library yang dibutuhkan...")
    !pip install minio trino
    from minio import Minio
    from trino.dbapi import connect

# Konfigurasi Koneksi
MINIO_ENDPOINT = "localhost:9005"
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "admin123"

TRINO_HOST = "localhost"
TRINO_PORT = 8080
TRINO_USER = "trino"

Library 'minio' dan 'trino' siap digunakan


## 1. Sanity Check (Prerequisites Validation)
Memeriksa apakah *container* Docker untuk `minio`, `hive-metastore`, dan `trino` sudah berjalan dengan baik.

In [15]:
def check_docker_service(service_name: str) -> bool:
    # cek apakah container docker sedang berjalan
    try:
        result = subprocess.run(
            ["docker", "ps", "--filter", f"name={service_name}", "--format", "{{.Names}}"],
            capture_output=True, text=True, check=True
        )
        return service_name in result.stdout
    except Exception as e:
        print(f"Error checking docker: {e}")
        return False

services = ["minio", "hive-metastore", "trino"]
all_running = True

for service in services:
    if check_docker_service(service):
        print(f"[OK] {service:20} sedang berjalan")
    else:
        print(f"[X]  {service:20} TIDAK berjalan!")
        all_running = False

if not all_running:
    print("\nPERINGATAN: Ada service yang belum jalan. Jalankan 'docker-compose up -d' dulu")
else:
    print("\nSemua service prerequisites sudah siap.")

[OK] minio                sedang berjalan
[OK] hive-metastore       sedang berjalan
[OK] trino                sedang berjalan

Semua service prerequisites sudah siap.


## 2. Upload CSV & Cleanup Previous Data
Menjalankan script upload CSV ke MinIO dan membersihkan *bucket* `iceberg` dari data eksperimen sebelumnya agar lokasi tabel benar-benar bersih.

In [16]:
# Upload CSV ke MinIO (Memanggil script eksternal)
print("Uploading CSV to MinIO...")
upload_result = subprocess.run(["python", "../code/upload_csv_to_lakehouse.py"], capture_output=True, text=True)
print(upload_result.stdout)
if upload_result.stderr:
    print(f"Errors (if any): {upload_result.stderr}")

# Cleanup Iceberg Bucket di MinIO
print("\nCleaning up previous Iceberg objects...")
client = Minio(
    endpoint=MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

bucket_name = "iceberg"
table_prefixes = [f"tpch/{t}/" for t in ["customer", "lineitem", "nation", "orders", "part", "partsupp", "region", "supplier"]]

if client.bucket_exists(bucket_name):
    removed_objects = 0
    for prefix in table_prefixes:
        # cari file dengan prefix tersebut
        objects_to_delete = [obj.object_name for obj in client.list_objects(bucket_name, prefix=prefix, recursive=True)]
        if objects_to_delete:
            for obj_name in objects_to_delete:
                client.remove_object(bucket_name, obj_name)
            removed_objects += len(objects_to_delete)
            print(f"  [-] Menghapus {len(objects_to_delete)} file dari prefix {prefix}")
    
    print(f"Total file Iceberg lama yang dihapus: {removed_objects} files.")
else:
    print(f"Bucket '{bucket_name}' belum ada, melewati proses cleanup.")

Uploading CSV to MinIO...
Project directory: /mnt/c/Users/Muzaraar/Documents/Kuliah/Kelas_Kuliah/Semester_6/BigData/BigData-Kelompok5
CSV directory:     /mnt/c/Users/Muzaraar/Documents/Kuliah/Kelas_Kuliah/Semester_6/BigData/BigData-Kelompok5/data/csv

Connecting to MinIO localhost:9005...
Connected
  Bucket exists: lakehouse

Found 8 CSV file(s)
Uploading to s3://lakehouse/csv/

Cleaning up old files in s3://lakehouse/csv/...
Removed 1 old object(s).

  Uploaded customer.csv         ( 235.10 MB) → csv/customer/customer.csv
  Uploaded lineitem.csv         (   7.25 GB) → csv/lineitem/lineitem.csv
  Uploaded nation.csv           (   2.19 KB) → csv/nation/nation.csv
  Uploaded orders.csv           (   1.63 GB) → csv/orders/orders.csv
  Uploaded part.csv             ( 232.25 MB) → csv/part/part.csv
  Uploaded partsupp.csv         (   1.13 GB) → csv/partsupp/partsupp.csv
  Uploaded region.csv           (   0.38 KB) → csv/region/region.csv
  Uploaded supplier.csv         (  13.62 MB) → csv/su

## 3. Eksekusi Schema & Data Ingestion (Trino)
Membaca file `tpch_iceberg_schema.sql` dan mengirimkan perintah pembuatannya (*schema*, *external table*, *iceberg table*, dan *insert*) ke mesin komputasi Trino.

In [17]:
sql_script_path = "../code/tpch_iceberg_schema.sql"

try:
    with open(sql_script_path, 'r') as f:
        sql_content = f.read()
    
    # connect ke Trino
    conn = connect(
        host=TRINO_HOST, port=TRINO_PORT, user=TRINO_USER, 
        catalog="hive", schema="default", request_timeout=600
    )
    cursor = conn.cursor()
    print("Berhasil terhubung ke Trino UI (localhost:8080). Mulai mengeksekusi SQL...\n")
    
    # pisahkan perintah SQL berdasarkan titik koma (;)
    raw_statements = [stmt.strip() for stmt in sql_content.split(';') if stmt.strip()]
    
    for i, stmt in enumerate(raw_statements, 1):
        try:
            # log
            short_stmt = stmt[:60].replace('\n', ' ') + "..."
            print(f"[{i}/{len(raw_statements)}] Mengeksekusi: {short_stmt}")
            
            cursor.execute(stmt)
            
            # Jika iquery INSERT, beri jeda nafas untuk Trino
            if "INSERT INTO" in stmt.upper():
                print("    -> Ingestion sedang berjalan (cek Trino UI). Menunggu selesai...")
                time.sleep(2)
                
        except Exception as e:
            # Mengabaikan error wajar seperti "Tabel sudah ada"
            if "already exists" not in str(e).lower():
                print(f"    -> ERROR: {str(e)[:100]}")
                
    cursor.close()
    conn.close()
    print("\nSemua SQL Statements selesai dieksekusi!")
    
except Exception as e:
    print(f"Gagal mengeksekusi SQL via Trino: {e}")

Berhasil terhubung ke Trino UI (localhost:8080). Mulai mengeksekusi SQL...

[1/47] Mengeksekusi: -- Create schema for TPC-H CREATE SCHEMA IF NOT EXISTS icebe...
[2/47] Mengeksekusi: -- Create schema for external CSV views in Hive catalog CREA...
[3/47] Mengeksekusi: -- Create external table views over CSV files CREATE TABLE I...
[4/47] Mengeksekusi: CREATE TABLE IF NOT EXISTS hive.tpch_external.lineitem (    ...
[5/47] Mengeksekusi: CREATE TABLE IF NOT EXISTS hive.tpch_external.nation (     n...
[6/47] Mengeksekusi: CREATE TABLE IF NOT EXISTS hive.tpch_external.orders (     o...
[7/47] Mengeksekusi: CREATE TABLE IF NOT EXISTS hive.tpch_external.part (     p_p...
[8/47] Mengeksekusi: CREATE TABLE IF NOT EXISTS hive.tpch_external.partsupp (    ...
[9/47] Mengeksekusi: CREATE TABLE IF NOT EXISTS hive.tpch_external.region (     r...
[10/47] Mengeksekusi: CREATE TABLE IF NOT EXISTS hive.tpch_external.supplier (    ...
[11/47] Mengeksekusi: -- Drop existing Iceberg tables to reset metadata c

## 4. Validasi Ingestion (Row Counts)
Menghitung jumlah baris data yang berhasil disalin dari file mentah ke tabel berformat Parquet/Iceberg.

In [18]:
tables = ["customer", "lineitem", "nation", "orders", "part", "partsupp", "region", "supplier"]
total_rows = 0

try:
    # read katalog Iceberg
    conn = connect(
        host=TRINO_HOST, port=TRINO_PORT, user=TRINO_USER, 
        catalog="iceberg", schema="tpch", request_timeout=600
    )
    cursor = conn.cursor()
    
    print(f"{'NAMA TABEL':<15} | {'JUMLAH BARIS':>15} | {'STATUS'}")
    print("-" * 45)
    
    for table in tables:
        try:
            cursor.execute(f"SELECT COUNT(*) FROM iceberg.tpch.{table}")
            count = cursor.fetchone()[0]
            total_rows += count
            status = "OK" if count > 0 else "KOSONG"
            
            print(f"{table:<15} | {count:>15,} | {status}")
        except Exception as e:
            print(f"{table:<15} | {'ERROR':>15} | {str(e)[:30]}")
            
    print("-" * 45)
    print(f"Total Rows Ingested : {total_rows:,}")
    
    cursor.close()
    conn.close()
    
except Exception as e:
    print(f"Koneksi Trino untuk validasi gagal: {e}")

NAMA TABEL      |    JUMLAH BARIS | STATUS
---------------------------------------------
customer        |       1,500,000 | OK
lineitem        |      59,986,052 | OK
nation          |              25 | OK
orders          |      15,000,000 | OK
part            |       2,000,000 | OK
partsupp        |       8,000,000 | OK
region          |               5 | OK
supplier        |         100,000 | OK
---------------------------------------------
Total Rows Ingested : 86,586,082
